## 01b - Anonymization

Replaces each review id with a keyed hash, masks ticket-like codes in the review text and drops the author names.
It needs a secret key in `.env` (see `.env.example`), and every later notebook starts from its output.

Input: `data/raw/trenord_reviews_raw.csv` and `PSEUDO_KEY` from `.env`

Output: `data/processed/trenord_reviews_anon.csv`

In [1]:
# --- IMPORTS ---
import hashlib
import hmac
import os
import re

import pandas as pd
from dotenv import find_dotenv, load_dotenv

# --- LOAD SECRET KEY ---
# PSEUDO_KEY is the secret key of the keyed hash in step 1. It lives only in .env at the
# repo root (gitignored); .env.example documents the variable with a placeholder.
# It was generated with: python -c "import secrets; print(secrets.token_hex(32))"
# (64 hex characters, i.e. 256 random bits).
# find_dotenv(usecwd=True) searches for .env from the current working directory (notebooks/)
# upwards; load_dotenv() adds its variables to the environment, where os.getenv() reads them.
load_dotenv(find_dotenv(usecwd=True))
PSEUDO_KEY = os.getenv("PSEUDO_KEY")

# Stop before hashing if the key is missing, empty or still the placeholder:
# ids hashed with a known key could be recomputed by anyone from the public store ids.
if PSEUDO_KEY is None or PSEUDO_KEY.strip() == "" or PSEUDO_KEY == "change_me":
    raise RuntimeError("PSEUDO_KEY not found: create .env from .env.example with a real key.")

# hmac needs the key as bytes; encoded once here rather than at every call.
KEY_BYTES = PSEUDO_KEY.encode("utf-8")

# --- LOAD RAW DATA ---
# The raw file is only read; all changes are applied to a copy.
raw_df = pd.read_csv("../data/raw/trenord_reviews_raw.csv")
df = raw_df.copy()

print(f"Loaded {len(df):,} reviews, {df.shape[1]} columns")

Loaded 7,329 reviews, 10 columns


In [2]:
# --- STEP 1: PSEUDONYMISE review_id ---
# HMAC-SHA256 hashes the id together with the secret key. In plain words:
# - the same id always gives the same code, so a review keeps its pseudonym across runs;
# - without the key the code cannot be recomputed from the public store id,
#   so published rows cannot be matched back to the store.
# Only the first 12 hex characters (48 bits) of the 64 are kept: short, yet the chance of
# any collision among ~7,300 ids is about 1 in 10 million (uniqueness is checked below).

def pseudonymize_id(original_id):
    message_bytes = str(original_id).encode("utf-8")
    digest = hmac.new(KEY_BYTES, message_bytes, hashlib.sha256).hexdigest()
    return digest[:12]

df["review_id"] = df["review_id"].apply(pseudonymize_id)

print(f"Pseudonymised ids: {df['review_id'].nunique():,} unique out of {len(df):,}")

Pseudonymised ids: 7,329 unique out of 7,329


In [3]:
# --- STEP 2: MASK CODES IN REVIEW TEXT ---
# Long alphanumeric strings are typically ticket, order or card numbers.
# Mixed letters+digits (8+ chars) are masked first, then pure digit runs (8+ digits).

MIXED_CODE_PATTERN = re.compile(r"\b(?=[A-Za-z0-9]*\d)(?=[A-Za-z0-9]*[A-Za-z])[A-Za-z0-9]{8,}\b")
DIGIT_CODE_PATTERN = re.compile(r"(?<!\d)\d{8,}(?!\d)")

masked_texts = []
code_replacements = 0
for text in df["text"]:
    if not isinstance(text, str):
        masked_texts.append(text)
        continue
    text, mixed_count = MIXED_CODE_PATTERN.subn("[CODE]", text)
    text, digit_count = DIGIT_CODE_PATTERN.subn("[CODE]", text)
    code_replacements += mixed_count + digit_count
    masked_texts.append(text)
df["text"] = masked_texts

print(f"[CODE] replacements in text: {code_replacements}")

[CODE] replacements in text: 5


In [4]:
# --- STEP 3: DROP author ---
# Reviewer names are direct identifiers and are not needed downstream.
df = df.drop(columns=["author"])

print(f"Columns kept: {df.columns.tolist()}")

Columns kept: ['review_id', 'rating', 'text', 'thumbs_up', 'app_version', 'date', 'has_reply', 'reply_text', 'reply_date']


In [5]:
# --- CHECKS (counts only) ---
original_ids = set(raw_df["review_id"].astype(str))
pseudonymised_ids = set(df["review_id"].astype(str))

# reply_text must be untouched: compared row by row with the raw file (same row order).
# Two missing values count as identical.
replies_different = 0
for raw_reply, anon_reply in zip(raw_df["reply_text"], df["reply_text"]):
    both_missing = pd.isna(raw_reply) and pd.isna(anon_reply)
    if not both_missing and raw_reply != anon_reply:
        replies_different += 1

checks = {
    "same row count as raw": len(df) == len(raw_df),
    "author column absent": "author" not in df.columns,
    "review_id unique": df["review_id"].is_unique,
    "no original review_id present": len(original_ids & pseudonymised_ids) == 0,
    "reply_text identical to raw for every row": replies_different == 0,
}

print(f"Rows raw / anon                       : {len(raw_df):,} / {len(df):,}")
print(f"Replies (non-null) raw / anon         : {raw_df['reply_text'].notna().sum():,} / {df['reply_text'].notna().sum():,}")
print(f"Replies different from raw            : {replies_different}")
print(f"[CODE] replacements                   : {code_replacements}")
print()
for check_name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {check_name}")

if not all(checks.values()):
    raise AssertionError("Anonymisation checks failed - output not saved.")

Rows raw / anon                       : 7,329 / 7,329
Replies (non-null) raw / anon         : 3,052 / 3,052
Replies different from raw            : 0
[CODE] replacements                   : 5

PASS  same row count as raw
PASS  author column absent
PASS  review_id unique
PASS  no original review_id present
PASS  reply_text identical to raw for every row


In [6]:
# --- SAVE ANONYMISED DATA ---
df.to_csv("../data/processed/trenord_reviews_anon.csv", index=False, encoding="utf-8-sig")

print(f"Saved {len(df):,} reviews to ../data/processed/trenord_reviews_anon.csv")

Saved 7,329 reviews to ../data/processed/trenord_reviews_anon.csv


In [7]:
# --- PREVIEW ---
# Text and reply columns are left out on purpose: notebook outputs are published with the
# repository, so they show only pseudonymised ids and non-text fields.
df[["review_id", "rating", "thumbs_up", "app_version", "date", "has_reply"]].head()

,review_id,rating,thumbs_up,app_version,date,has_reply
0,e0441c09f9ba,1,0,4.16.7,2026-08-04 16:01:12,False
1,13bff3758628,1,0,4.16.7,2026-08-03 17:29:22,False
2,525ab22d4722,2,0,5.0.7,2026-08-03 06:48:29,False
3,525d76d975fa,1,51,5.0.7,2026-08-02 22:42:05,False
4,20578ee8ad99,1,0,4.16.7,2026-07-31 21:30:14,False
